# pyQuil Tutorial: Programs and Gates
*Condensed from the [pyQuil docs](https://pyquil-docs.rigetti.com/en/stable/programs_and_gates.html).*

If you haven’t built the **Python (pyQuil container)** kernel and run the **quilc (quilc -P -S)** and **qvm (qvm -S)** servers yet, run the **0_Jupyter_Pykernel_from_Container_image** notebook to create it from pyquil.sif, then select **Python (pyQuil container)** in the top-right corner (refresh the page if it isn’t listed). Cells that call `get_qc()`, `compile()` or `run()` need the `quilc` and `qvm` servers running:

In [2]:
import sys, pyquil
from collections import Counter
print(sys.executable, pyquil.__version__)

/opt/venv/bin/python 4.22.0


In [3]:
!tail -n 3 quilc.log qvm.log

==> quilc.log <==
<134>1 2026-10-26T16:45:10Z r240.ib.bridges2.psc.edu quilc 5024 LOG0002 [rigetti@0000 methodName="get_version_info" requestID="c2390b94-c7bc-4c55-9d99-7956dc33a194" wallTime="0.002" error="false"] Requested get_version_info completed
<134>1 2026-10-26T16:45:10Z r240.ib.bridges2.psc.edu quilc 5024 - - Request e494939b-0fb7-4165-bcbb-1b9182e5eeee received for quil_to_native_quil
<134>1 2026-10-26T16:45:11Z r240.ib.bridges2.psc.edu quilc 5024 LOG0002 [rigetti@0000 methodName="quil_to_native_quil" requestID="e494939b-0fb7-4165-bcbb-1b9182e5eeee" wallTime="0.316" error="false"] Requested quil_to_native_quil completed

==> qvm.log <==
<135>1 2026-10-26T16:45:11Z r240.ib.bridges2.psc.edu qvm 4885 - - [127.0.0.1 Session:3] Finished in 24 ms
<134>1 2026-10-26T16:45:11Z r240.ib.bridges2.psc.edu qvm 4885 - - 127.0.0.1 - [2026-10-26 12:45:11] "POST / HTTP/1.1" 200 68 "-" "-"



In [4]:
!cat qvm.log quilc.log

******************************
* Welcome to the Rigetti QVM *
******************************
Copyright (c) 2016-2019 Rigetti Computing.

This is a part of the Forest SDK. By using this program
you agree to the End User License Agreement (EULA) supplied
with this program. If you did not receive the EULA, please
contact <support@rigetti.com>.

(Configured with 10240 MiB of workspace and 128 workers.)
(Gates parallelize at 19 qubits.)
(There are 3 kernels and they are used with up to 29 qubits.)
(Features enabled: none)

<134>1 2026-10-26T16:44:57Z r240.ib.bridges2.psc.edu qvm 4885 - - Compilation mode disabled.
<135>1 2026-10-26T16:44:57Z r240.ib.bridges2.psc.edu qvm 4885 - - Selected simulation method: pure-state
<135>1 2026-10-26T16:44:57Z r240.ib.bridges2.psc.edu qvm 4885 - - Starting server on port 5000.
<134>1 2026-10-26T16:44:57Z r240.ib.bridges2.psc.edu qvm 4885 LOG0001 - This is the latest version of the SDK.
<135>1 2026-10-26T16:45:02Z r240.ib.bridges2.psc.edu qvm 4885 - - [127.

## 1. Introduction
Qubits start in the ground state |0⟩. `X` flips qubit 0 to |1⟩. Printing a program shows its Quil code.

In [5]:
import warnings
from pyquil._deprecation import PyQuilDeprecationWarning
# Hide pyQuil 4.22's notices that the QVM is deprecated in favour of pyQuil v5
# (the QVM still works; see 0_Jupyter_Pykernel_from_Container_image.ipynb).
warnings.filterwarnings("ignore", category=PyQuilDeprecationWarning)

from pyquil import Program, get_qc
from pyquil.gates import *
import numpy as np

p = Program()
p += X(0)
print(p)

X 0



To read results, **declare** classical memory and **measure** into it:

In [6]:
p = Program()
ro = p.declare('ro', 'BIT', 1)
p += X(0)
p += MEASURE(0, ro[0])
print(p)

DECLARE ro BIT[1]
X 0
MEASURE 0 ro[0]



In [7]:
#print(get_qc.__doc__)

Run on the Quantum Virtual Machine (QVM) — expect `[[1]]`:

In [8]:
qc = get_qc('1q-qvm')
executable = qc.compile(p)
result = qc.run(executable)
print(result.get_register_map().get('ro'))

[[1]]


## 2. Standard Gate Set
| Type | Gates |
|---|---|
| Pauli | `I`, `X`, `Y`, `Z` |
| Hadamard | `H` |
| Phase | `PHASE(θ)`, `S`, `T` |
| Controlled phase | `CZ`, `XY`, `CPHASE00(α)`, `CPHASE01(α)`, `CPHASE10(α)`, `CPHASE(α)` |
| Rotations | `RX(θ)`, `RY(θ)`, `RZ(θ)` |
| Controlled X | `CNOT`, `CCNOT` |
| Swap | `SWAP`, `CSWAP`, `ISWAP`, `PSWAP(α)` |

Parametric gates take float arguments.

## 3. Declaring Memory
Classical memory must be declared: `p.declare(name, memory_type='BIT', memory_size=1, shared_region=None, offsets=None)`.
- `memory_type`: `'BIT'`, `'REAL'`, `'OCTET'`, or `'INTEGER'`
- `shared_region` / `offsets`: memory aliasing

⚠️ `.declare()` returns a memory reference, not the Program, so it can't be chained.

In [9]:
p = Program()
ro = p.declare('ro', 'BIT', 16)
theta = p.declare('theta', 'REAL')
print(p)

DECLARE ro BIT[16]
DECLARE theta REAL[1]



## 4. Measurement
Use `MEASURE(qubit, memory_ref)`. Example: a Bell state measured on two qubits.

In [10]:
p = Program()
ro = p.declare('ro', 'BIT', 2)
p += H(0)
p += CNOT(0, 1)
p += MEASURE(0, ro[0])
p += MEASURE(1, ro[1])
print(p)

DECLARE ro BIT[2]
H 0
CNOT 0 1
MEASURE 0 ro[0]
MEASURE 1 ro[1]



Measuring a list of qubits in a loop:

In [11]:
p = Program()
qubits = [5, 6, 7]
ro = p.declare('ro', 'BIT', len(qubits))
for i, q in enumerate(qubits):
    p += MEASURE(q, ro[i])
print(p)

DECLARE ro BIT[3]
MEASURE 5 ro[0]
MEASURE 6 ro[1]
MEASURE 7 ro[2]



### Number of trials (shots)
`.wrap_in_numshots_loop(n)` runs the program `n` times.

In [12]:
p = Program()
ro = p.declare('ro', 'BIT', 2)
p += H(0)
p += CNOT(0, 1)
p += MEASURE(0, ro[0])
p += MEASURE(1, ro[1])
p.wrap_in_numshots_loop(1000)

qc = get_qc('2q-qvm')
bits = qc.run(qc.compile(p)).get_register_map().get('ro')
counts = dict(Counter("".join(map(str, row)) for row in bits))
print(counts) 

{'11': 484, '00': 516}


### Fixed-count loop in Quil
`.with_loop()` wraps the body in a Quil-level loop (using a counter register and labels) instead of a shot loop.

In [13]:
from pyquil.quilatom import Label

p = Program(H(0), CNOT(0, 1))
ro = p.declare("ro", "BIT", 2)
p.measure(0, ro[0])
p.measure(1, ro[1])
shot_count = p.declare("shot_count", "INTEGER")

looped_program = p.with_loop(1000, shot_count, Label("start-loop"), Label("end-loop"))
print(looped_program.out())

qc = get_qc("2q-qvm")
results = qc.run(looped_program).get_register_map().get('ro')
counts = dict(Counter("".join(map(str, row)) for row in results))
print(counts)

DECLARE ro BIT[2]
DECLARE shot_count INTEGER[1]
MOVE shot_count[0] 1000
LABEL @start-loop
H 0
CNOT 0 1
MEASURE 0 ro[0]
MEASURE 1 ro[1]
SUB shot_count[0] 1
JUMP-UNLESS @end-loop shot_count[0]
JUMP @start-loop
LABEL @end-loop

{'10': 1}


## 5. Parametric Compilation
Compile a program **template** once, then supply parameter values at run time — much faster than recompiling for every value. Unset memory defaults to zero.

In [14]:
qubit = 0
p = Program()
ro = p.declare("ro", "BIT", 1)
theta_ref = p.declare("theta", "REAL")

p += RX(np.pi / 2, qubit)
p += RZ(theta_ref, qubit)
p += RX(-np.pi / 2, qubit)
p += MEASURE(qubit, ro[0])

In [15]:
qc = get_qc("1q-qvm")
executable = qc.compile(p)

# one memory map per theta value
memory_maps = [{"theta": [theta]} for theta in np.linspace(0, 2 * np.pi, 200)]
#print(memory_maps)
parametric_measurements = qc.run_with_memory_map_batch(executable, memory_maps)
print(len(parametric_measurements))

200


## 6. Gate Modifiers
- **`DAGGER`** – adjoint: `DAGGER RX(pi/3) 0` ≡ `RX(-pi/3) 0`
- **`CONTROLLED`** – adds a control qubit: `CCNOT 0 1 2` ≡ `CONTROLLED CNOT 0 1 2` ≡ `CONTROLLED CONTROLLED X 0 1 2`
- **`FORKED`** – picks parameters based on a control qubit `c`: uses `G(u…)` if `c=0`, `G(v…)` if `c=1`

In Python, use `.dagger()`, `.controlled()`, `.forked()`:

In [16]:
print(Program(RX(np.pi/3, 0).dagger()))
print(Program(X(1).controlled(0)))
print(Program(X(2).controlled(0).controlled(1)))
print(Program(X(2).controlled([0, 1])))
print(Program(X(0), X(1).controlled(0), X(0)))
print(Program(RX(np.pi/2, 1).forked(0, [np.pi])))

DAGGER RX(1.0471975511965976) 0

CONTROLLED X 0 1

CONTROLLED CONTROLLED X 1 0 2

CONTROLLED CONTROLLED X 1 0 2

X 0
CONTROLLED X 0 1
X 0

FORKED RX(1.5707963267948966, 3.141592653589793) 0 1



## 7. Defining New Gates
Create a custom gate from a unitary matrix with `DefGate`, then get a constructor for it. The definition must be added to the program. We need to use both definition (sqrt_x_definition) and gate (SQRT-X) operation.

In [17]:
from pyquil.quil import DefGate

sqrt_x = np.array([[0.5+0.5j, 0.5-0.5j],
                   [0.5-0.5j, 0.5+0.5j]])

sqrt_x_definition = DefGate("SQRT-X", sqrt_x)
SQRT_X = sqrt_x_definition.get_constructor()

p = Program()
p += sqrt_x_definition
p += SQRT_X(0)
print(p)

DEFGATE SQRT-X AS MATRIX:
    0.5+0.5i, 0.5-0.5i
    0.5-0.5i, 0.5+0.5i

SQRT-X 0



Multi-qubit gates work the same way (here: `X ⊗ √X`):

In [18]:
x_gate_matrix = np.array([[0.0, 1.0], [1.0, 0.0]])
x_sqrt_x = np.kron(x_gate_matrix, sqrt_x)

x_sqrt_x_definition = DefGate("X-SQRT-X", x_sqrt_x)
X_SQRT_X = x_sqrt_x_definition.get_constructor()

p = Program(x_sqrt_x_definition, X_SQRT_X(0, 1))
print(p)

DEFGATE X-SQRT-X AS MATRIX:
    0, 0, 0.5+0.5i, 0.5-0.5i
    0, 0, 0.5-0.5i, 0.5+0.5i
    0.5+0.5i, 0.5-0.5i, 0, 0
    0.5-0.5i, 0.5+0.5i, 0, 0

X-SQRT-X 0 1



### Parametric gates
Use `Parameter` and Quil functions (`quil_sin`, `quil_cos`, `quil_sqrt`, `quil_exp`, `quil_cis`) in the matrix. Example: controlled-RX.

In [19]:
from pyquil.quilatom import Parameter, quil_sin, quil_cos
from pyquil.quilbase import DefGate

theta = Parameter('theta')
crx = np.array([
    [1, 0, 0, 0],
    [0, 1, 0, 0],
    [0, 0, quil_cos(theta / 2), -1j * quil_sin(theta / 2)],
    [0, 0, -1j * quil_sin(theta / 2), quil_cos(theta / 2)]
])

gate_definition = DefGate('CRX', crx, [theta])
CRX = gate_definition.get_constructor()

p = Program()
p += gate_definition
p += H(0)
p += CRX(np.pi/2)(0, 1)
print(p)

DEFGATE CRX(%theta) AS MATRIX:
    1, 0, 0, 0
    0, 1, 0, 0
    0, 0, cos(%theta/2), -1.0i*sin(%theta/2)
    0, 0, -1.0i*sin(%theta/2), cos(%theta/2)

H 0
CRX(1.5707963267948966) 0 1



### Permutation gates
Gates that just permute basis states can be written compactly with `DefPermutationGate` instead of a full matrix. Both below define CCNOT:

In [20]:
from pyquil.quilbase import DefPermutationGate

ccnot_matrix = np.eye(8, dtype=int)
ccnot_matrix[[6, 7]] = ccnot_matrix[[7, 6]]
ccnot_gate = DefGate("MATRIX_CCNOT", ccnot_matrix)

ccnot_perm = DefPermutationGate("PERMUTATION_CCNOT", [0, 1, 2, 3, 4, 5, 7, 6])
print(Program(ccnot_perm))

DEFGATE PERMUTATION_CCNOT AS PERMUTATION:
    0, 1, 2, 3, 4, 5, 7, 6




## 8. Pragmas
`PRAGMA` directives control how programs are compiled/simulated without changing their meaning.

### Qubit rewiring
The compiler may relabel qubits to fit the hardware. `INITIAL_REWIRING` sets the strategy:
- `"GREEDY"` – let the compiler choose good qubits
- `"NAIVE"` – keep your original qubit labels

In [21]:
from pyquil.quil import Pragma

p = Program(Pragma('INITIAL_REWIRING', ['"GREEDY"']))
p += X(3)
print(p)

p_naive = Program(Pragma('INITIAL_REWIRING', ['"NAIVE"']), X(3))
print(p_naive)

PRAGMA INITIAL_REWIRING "GREEDY"
X 3

PRAGMA INITIAL_REWIRING "NAIVE"
X 3



## 9. Delays
`DELAY(qubit, seconds)` inserts a wait. ⚠️ `DELAY` (and other Quil-T instructions) aren't supported by QVM/quilc — strip them when simulating. (`PRAGMA DELAY` is deprecated in v4.)

In [22]:
from pyquil.api import QVM

qc = get_qc("2q-qvm")
p = Program(H(0))
p += DELAY(0, 200e-9)

if isinstance(qc.qam, QVM):
    p = p.remove_quil_t_instructions()
else:
    p = qc.compiler.native_quil_to_executable(p)

qc.run(p)

QAMExecutionResult(executable=Program { calibrations: Calibrations { calibrations: CalibrationSet { data: [] }, measure_calibrations: CalibrationSet { data: [] } }, extern_pragma_map: ExternPragmaMap({}), frames: FrameSet { frames: {} }, memory_regions: {}, waveforms: {}, gate_definitions: {}, instructions: [Gate(Gate { name: "H", parameters: [], qubits: [Fixed(0)], modifiers: [] })], used_qubits: {Fixed(0)} }, data=ExecutionData { result_data: Qvm(QvmResultData { memory: {} }), duration: None })

## 10. Ways to Construct Programs
All of the following produce the same program (`X 0; Y 1`). Using `+=` is preferred.

In [23]:
p = Program(); p += X(0); p += Y(1); print(p)      # preferred
print(Program(X(0), Y(1)))                          # instructions in constructor
print(Program(X(0)) + Program(Y(1)))                # composing programs
print(Program("X 0\nY 1"))                         # raw Quil, newline
print(Program("X 0", "Y 1"))                        # raw Quil, comma-separated
print(Program().inst(X(0)).inst(Y(1)))              # chained .inst (less preferred)

X 0
Y 1

X 0
Y 1

X 0
Y 1

X 0
Y 1

X 0
Y 1

X 0
Y 1



### Kill the servers after use in **0_Jupyter_Pykernel_from_Container_image** notebook